# Python OOP for ML

Use small estimator classes to separate configuration, fitted state, reusable behavior, and composition.

- **Study time:** 30-40 minutes
- **Prerequisites:** functions, classes, NumPy, and train/test vocabulary
- **Mode:** `quick`
- **Data policy:** no external files or downloads; seeded synthetic arrays only
- **Provenance:** new connective material built around the cleaned legacy algorithms

Output convention: every retained textual result begins with a label that identifies the operation that produced it.
Annotation convention: comments explain intent, shape changes, invariants, subtle API behavior, or configuration side effects; obvious Python syntax is left uncommented.


In [1]:
from dataclasses import dataclass

import numpy as np

from datacoding.algorithms import KMeans, KNNClassifier, LinearRegressionGD
from datacoding.algorithms._validation import NotFittedError

rng = np.random.default_rng(11)  # Reproducible local generator without global RNG side effects.

## 1. Configuration versus learned state

Constructor fields describe choices. Attributes ending in `_` are learned during `fit`.


In [2]:
model = LinearRegressionGD(learning_rate=0.08, max_iter=2_000, l2=0.001)
configuration = {
    key: value for key, value in vars(model).items() if not key.endswith("_")
}  # By convention, trailing-underscore attributes are learned state rather than configuration.
print("Estimator | constructor configuration", configuration)

X = rng.normal(size=(200, 2))
y = 2.0 * X[:, 0] - 1.5 * X[:, 1] + 0.7
model.fit(X, y)  # Populate learned attributes such as coef_ and intercept_.

learned = {
    "coef_": model.coef_,
    "intercept_": model.intercept_,
    "n_features_in_": model.n_features_in_,
}
print("Estimator | learned state after fit", learned)

Estimator | constructor configuration {'learning_rate': 0.08, 'max_iter': 2000, 'l2': 0.001, 'tolerance': 1e-10}
Estimator | learned state after fit {'coef_': array([ 1.99783114, -1.49836223]), 'intercept_': 0.7000719904572189, 'n_features_in_': 2}


## 2. A focused transformer class


In [3]:
@dataclass  # Generate the constructor for explicit hyperparameter fields only.
class Standardizer:
    epsilon: float = 1e-12

    def fit(self, X):
        values = np.asarray(X, dtype=float)
        self.mean_ = values.mean(axis=0)  # Learned state: one statistic per feature.
        self.scale_ = values.std(axis=0) + self.epsilon  # Keep constant-feature transforms finite.
        return self  # Returning self enables estimator-style chaining.

    def transform(self, X):
        if not hasattr(self, "mean_"):
            raise NotFittedError("Standardizer must be fitted before transform")
        return (
            np.asarray(X, dtype=float) - self.mean_
        ) / self.scale_  # Reuse fitted statistics; never refit here.

    def fit_transform(self, X):
        return self.fit(X).transform(X)


standardizer = Standardizer()
X_standard = standardizer.fit_transform(X)
print("Transformer | standardized feature means", X_standard.mean(axis=0))
print("Transformer | standardized feature std", X_standard.std(axis=0))

Transformer | standardized feature means [-1.49880108e-17 -8.32667268e-18]
Transformer | standardized feature std [1. 1.]


## 3. Composition instead of deep inheritance


In [4]:
@dataclass  # Treat component objects as pipeline configuration.
class RegressionPipeline:
    transformer: Standardizer
    estimator: LinearRegressionGD

    def fit(self, X, y):
        transformed = self.transformer.fit_transform(
            X
        )  # Learn preprocessing only during pipeline.fit.
        self.estimator.fit(transformed, y)
        return self

    def predict(self, X):
        return self.estimator.predict(self.transformer.transform(X))


pipeline = RegressionPipeline(
    transformer=Standardizer(),
    estimator=LinearRegressionGD(learning_rate=0.08, max_iter=2_000),
).fit(X, y)
pipeline_mse = np.mean((pipeline.predict(X) - y) ** 2)
print("Composition | pipeline training MSE sanity check", pipeline_mse)

Composition | pipeline training MSE sanity check 2.0404636847199593e-19


## 4. Different estimators, consistent interface


In [5]:
class_X = np.array([[0, 0], [0, 1], [1, 0], [9, 9], [9, 10], [10, 9]], dtype=float)
class_y = np.array(["near-zero"] * 3 + ["near-nine"] * 3)
knn = KNNClassifier(n_neighbors=3).fit(class_X, class_y)

cluster_X = np.vstack(
    [rng.normal([0, 0], 0.2, size=(40, 2)), rng.normal([4, 4], 0.2, size=(40, 2))]
)
kmeans = KMeans(n_clusters=2, random_state=11).fit(cluster_X)

print("Polymorphic pattern | KNN predictions", knn.predict([[0.2, 0.2], [9.5, 9.1]]))
print("Polymorphic pattern | K-means learned centers", kmeans.cluster_centers_)

Polymorphic pattern | KNN predictions ['near-zero' 'near-nine']
Polymorphic pattern | K-means learned centers [[ 7.78657425e-03 -1.10510185e-03]
 [ 3.99295471e+00  4.00029397e+00]]


## 5. Boundary errors should be explicit


In [6]:
try:
    Standardizer().transform([[1.0, 2.0]])
except NotFittedError as error:
    print("Expected error | transform before fit", type(error).__name__)

try:
    model.predict([[1.0, 2.0, 3.0]])
except ValueError as error:
    print("Expected error | feature-count mismatch", str(error))

print("OOP drill | status", "configuration, fitted state, composition, and validation demonstrated")

Expected error | transform before fit NotFittedError
Expected error | feature-count mismatch Prediction feature count does not match training data
OOP drill | status configuration, fitted state, composition, and validation demonstrated
